# Telecom-T2C — Inference Server (Drive adapter -> ngrok tunnel)

Loads a LoRA adapter that was fine-tuned and synced to Google Drive by
`Telecom_T2C_Trainer_v2.ipynb`, serves it over a local HTTP endpoint inside
this Colab runtime, and tunnels that endpoint out through
[ngrok](https://ngrok.com) so you can send requests to it from your own PC.

**Why this exists:** a 12B model (even 4-bit QLoRA) needs far more VRAM than
a typical laptop GPU has — see the project README's "Testing the adapter
locally" note. This notebook keeps the GPU work on Colab and lets you talk
to it from a normal Python/curl client on your own machine.

**This notebook only orchestrates** — the HTTP server itself lives in
`src/server.py`, model reload logic in `src/inference.py`. Adapters are
never merged into the base model.

**Prerequisites:**
- A free [ngrok](https://ngrok.com) account and its authtoken (from
  https://dashboard.ngrok.com/get-started/your-authtoken). Add it as a Colab
  secret named `NGROK_AUTHTOKEN` (key icon in the left sidebar), or paste it
  in when this notebook prompts for it.
- The trainer notebook must have already run its Section 11 (Save), which
  Drive-syncs `run_<timestamp>/adapter/` under
  `drive.google_drive_directory` (see `configs/experiment.yaml`).

Run cells top to bottom. Sections: Runtime Check, Install, Configuration,
Locate Adapter, Load Model, Local Smoke Test, Start Server, Connect From
Your PC, Stop Server.

## 1. Runtime Check

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Locate the Telecom-T2C project root from any Colab/local starting cwd."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for child in start.glob("*/"):
        if (child / "src").is_dir() and (child / "configs").is_dir():
            return child
    raise RuntimeError(
        "Could not locate the Telecom-T2C project root (a directory containing both "
        "'src/' and 'configs/'). If running in Colab, cd into the cloned/uploaded repo "
        "directory first, e.g.:\n  %cd /content/Telecom-T2C"
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
from src import utils

logger = utils.setup_logging()
gpu = utils.detect_gpu()
print(f"GPU: {gpu.name} | family={gpu.family} | vram={gpu.vram_gb:.1f} GB | bf16={gpu.bf16_supported}")

import torch

print(f"torch: {torch.__version__} | built for CUDA {torch.version.cuda}")


## 2. Install

Same phased install as `Telecom_T2C_Trainer_v2.ipynb` Section 2 (see that
notebook / `requirements.txt`'s top comment for the full reasoning —
unsloth/unsloth_zoo's PyPI metadata caps transformers below what
`google/gemma-4-12B-it` needs, so the correlated Unsloth stack and
transformers are installed in separate `--no-deps` phases). Duplicated here
rather than shared, since a notebook install cell can't import from a
not-yet-installed project. If you update the version floors in one
notebook, update the other to match.

Also installs `fastapi`, `uvicorn`, and `pyngrok` — only needed for serving,
not for training, so they're not in `requirements.txt`.

Re-run this cell after any runtime restart.

In [ ]:
import re
import subprocess
import sys

import torch

from src import utils

# Phase 1: everything that resolves normally (no --no-deps needed).
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-r", str(PROJECT_ROOT / "requirements.txt"),
    ]
)

# Phase 2: the correlated Unsloth ML stack, installed together with
# --no-deps — see Telecom_T2C_Trainer_v2.ipynb Section 2 for the full
# reasoning behind this split.
torch_version = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
xformers_pin = "xformers==" + {
    "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2",
}.get(torch_version, "0.0.34")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "unsloth_zoo", "bitsandbytes>=0.46.1,!=0.48.0", "accelerate>=1.8",
        xformers_pin, "peft>=0.19.1", "trl>=0.15.0", "triton", "unsloth",
    ]
)

# Phase 3: torchao, also --no-deps.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "torchao>=0.16.0",
    ]
)

# Phase 4: transformers + tokenizers, --no-deps, installed last.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "transformers==5.10.2", "tokenizers>=0.22.0,<=0.23.0",
    ]
)

# torchaudio: confirmed broken import chain on some Colab images, not
# needed by this text-only project — see Telecom_T2C_Trainer_v2.ipynb
# Section 2 for the full incident writeup.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
    check=False,
)

utils.disable_unused_transformers_backends()

# Serving-only dependencies (not needed for training, so not in requirements.txt).
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "fastapi", "uvicorn", "pyngrok"]
)

print("Install complete. If this is the first install in a fresh runtime, "
      "Runtime -> Restart session, then re-run from Section 1.")


## 3. Configuration

Loads `configs/experiment.yaml` for `model.base_model`, `data.max_seq_length`,
and `drive.google_drive_directory` — the same config the trainer notebook
used, so the adapter reloads against the exact base model it was trained
against.

In [ ]:
from src import config as config_mod

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
experiment_config = config_mod.load_config(CONFIG_PATH)
print(f"base_model: {experiment_config.model.base_model}")
print(f"max_seq_length: {experiment_config.data.max_seq_length}")
print(f"drive.google_drive_directory: {experiment_config.drive.google_drive_directory}")


## 4. Locate Adapter on Google Drive

Mounts Drive, then auto-detects the most recently synced `run_*/adapter/`
under `drive.google_drive_directory`. Set `ADAPTER_DIR_OVERRIDE` below (a
string path) if you want a specific run instead of the latest one.

In [ ]:
from pathlib import Path

from src import checkpoint as checkpoint_mod
from src import utils

ADAPTER_DIR_OVERRIDE = None  # e.g. "/content/drive/MyDrive/telecom_t2c/run_20260722_101500"

utils.mount_google_drive(experiment_config.drive.drive_mount_point)

if ADAPTER_DIR_OVERRIDE:
    adapter_dir = Path(ADAPTER_DIR_OVERRIDE) / "adapter"
else:
    drive_base = Path(experiment_config.drive.google_drive_directory)
    latest_run = checkpoint_mod.find_latest_synced_run(drive_base)
    if latest_run is None:
        raise RuntimeError(
            f"No synced run with an adapter/ directory found under {drive_base}. "
            "Run the trainer notebook through Section 11 (Save) first, or set "
            "ADAPTER_DIR_OVERRIDE above to a specific run directory."
        )
    adapter_dir = latest_run / "adapter"

print(f"Using adapter: {adapter_dir}")


## 5. Load Model + Adapter

Reloads the base model with the adapter attached, via the same
`inference.load_model_for_inference` the trainer notebook's Smoke Test
(Section 12) uses. Never merges the adapter.

In [ ]:
from src import inference as inference_mod
from src import tokenizer as tokenizer_mod

hf_token = tokenizer_mod.resolve_hf_token(experiment_config.model.hf_token_env_var)
inf_model, inf_tokenizer = inference_mod.load_model_for_inference(
    experiment_config.model, experiment_config.data.max_seq_length, str(adapter_dir), hf_token,
)


## 6. Local Smoke Test

Sanity-check generation inside the notebook before exposing anything
publicly — using a **real conversation from the validation set**, not a
hand-typed question. This matters: every training conversation starts with
a specific system prompt plus a "Deployment context" user turn before the
actual query (see README "Dataset format"). A bare one-line question with
no system prompt is out-of-distribution for this adapter — it'll just
answer like a generic assistant instead of producing PASS_0-4 output,
which looks like a bug but isn't one.

In [ ]:
import json as _json
from pathlib import Path

from src import utils

val_path = Path(experiment_config.data.val_path)
sample_row = next(iter(utils.read_jsonl(val_path)))
sample_messages = sample_row["messages"]
first_assistant_idx = next(i for i, m in enumerate(sample_messages) if m["role"] == "assistant")
prompt_messages = sample_messages[:first_assistant_idx]
gold = sample_messages[first_assistant_idx]["content"]

result = inference_mod.generate(inf_model, inf_tokenizer, prompt_messages, max_new_tokens=512)

print("=== PROMPT MESSAGES (system + context + query) ===")
print(_json.dumps(prompt_messages, indent=2)[:1500])
print()
print("=== GENERATED ===")
print(result[:1500])
print()
print("=== GOLD ===")
print(gold[:1500])


## 7. Start Inference Server + ngrok Tunnel

Starts a FastAPI server (`src/server.py`) on this Colab runtime and tunnels
it out through ngrok. The `/generate` endpoint requires the printed bearer
token — anyone with the ngrok URL *and* the token can reach your model, so
don't share either outside your own use. The token is generated fresh each
run and only ever printed here, never written to disk.

Keep this Colab tab open / the runtime active for as long as you want the
server reachable — an idle/disconnected Colab runtime kills the tunnel.

In [ ]:
from src import server as server_mod
from src import utils

ngrok_authtoken = utils.resolve_secret("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    from getpass import getpass

    ngrok_authtoken = getpass("Paste your ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken): ")

api_token = server_mod.generate_api_token()
app = server_mod.build_app(inf_model, inf_tokenizer, api_token, default_max_new_tokens=experiment_config.evaluation.max_new_tokens_eval)
sft_server, tunnel = server_mod.start_server(app, port=8000, ngrok_authtoken=ngrok_authtoken)

print(f"Public URL:   {tunnel.public_url}")
print(f"Bearer token: {api_token}")
print()
print("Example curl, using the SAME real prompt_messages from Section 6")
print("(a bare one-line question with no system prompt will just get a")
print("generic-assistant reply, not PASS_0-4 output — see Section 6's note):")
import json as _json

_curl_body = _json.dumps({"messages": prompt_messages})
print(
    f"curl -X POST {tunnel.public_url}/generate "
    f"-H 'Authorization: Bearer {api_token}' -H 'Content-Type: application/json' "
    f"-d '{_curl_body}'"
)


## 8. Connect From Your Local PC

From your own machine (no GPU/Colab needed), send requests with the public
URL and bearer token printed above. **`messages` must include the same
system prompt + "Deployment context" turn the adapter was trained on** —
see Section 6's `prompt_messages` for a real, working example (copy its
printed JSON), or README "Dataset format" for the exact shape. A bare
`{"role": "user", "content": "<question>"}` with nothing else will not
produce PASS_0-4 output — the model will just answer like a generic
assistant, since that shape never appeared in training.

```python
import requests

BASE_URL = "https://<your-ngrok-subdomain>.ngrok-free.app"  # from Section 7's output
API_TOKEN = "<paste bearer token from Section 7>"

# Replace this with the real prompt_messages printed by Section 6 —
# system prompt + deployment-context turn + your actual query.
messages = [
    {"role": "system", "content": "You are a GPON network inventory query compiler..."},
    {"role": "user", "content": "## Deployment context\n\n..."},
    {"role": "user", "content": "## Query\nList all OLT devices in the network."},
]

response = requests.post(
    f"{BASE_URL}/generate",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
    json={"messages": messages},
    timeout=120,
)
response.raise_for_status()
print(response.json()["generated_text"])
```

`GET {BASE_URL}/health` (no auth required) is a quick way to check the
tunnel is up before sending a real request.

## 9. Stop Server

Run when you're done — closes the ngrok tunnel and stops the local server.
Free ngrok tunnels also close automatically when this Colab runtime
disconnects.

In [ ]:
server_mod.stop_server(sft_server, tunnel)
print("Server stopped.")
